# Testing Vessel Delay Prediction Model 

model - Random Forest

Author - Yan

## Predict on New Vessels

In [6]:
import pandas as pd
import numpy as np
import joblib

In [7]:
## Predict whether a vessel will be delayed.

def predict_delay(speed, day_of_week, eta_hour, eta_min, week_number,
                  cog, draught, latitude, longitude,
                  speed_cat, vessel_type, direction, nav_status):

    # Load saved model and features
    saved_model    = joblib.load("vessel_delay_model.pkl")
    saved_features = joblib.load("vessel_delay_features.pkl")
 
    input_df = pd.DataFrame([{
        "speed"              : speed,
        "day_of_week"        : day_of_week,
        "eta_hour"           : eta_hour,
        "eta_min"            : eta_min,
        "week_number"        : week_number,
        "cog"                : cog,
        "maximim_draught"    : draught,
        "latitude"           : latitude,
        "longitude"          : longitude,
        "speed_cat"          : speed_cat,
        "vessel_type_simple" : vessel_type,
        "direction"          : direction,
        "navigational_status": nav_status,
    }])
 
    input_df = pd.get_dummies(
        input_df,
        columns=["speed_cat", "vessel_type_simple", "direction", "navigational_status"],
        drop_first=True,
    )
    
    # Align to training feature columns
    input_df = input_df.reindex(columns=saved_features, fill_value=0)
 
    pred = saved_model.predict(input_df)[0]
    prob = saved_model.predict_proba(input_df)[0][1]
    return pred, round(prob, 3)

In [8]:
# Example prediction

pred, prob = predict_delay(
    speed=0.3,
    day_of_week=4,          # Friday
    eta_hour=14,
    eta_min=30,
    week_number=42,
    cog=180.0,
    draught=7.2,
    latitude=1.27,
    longitude=103.87,
    speed_cat="moored",
    vessel_type="cargo",
    direction="S",
    nav_status="Moored",
)
 
print(f"\nExample prediction:")
print(f"  Result      : {'DELAYED' if pred == 1 else 'ON TIME'}")
print(f"  Delay prob  : {prob*100:.1f}%")


Example prediction:
  Result      : ON TIME
  Delay prob  : 48.0%
